# Assessment 1: Data Preprocessing & Exploration

### Dataset
U.S. Airline On-Time Performance 1987–2020 (2M row sample, BTS)

### Business Question
What factors are most predictive of flight arrival delays, and how have delay patterns shifted across airlines, airports, and seasons over the 33-year period from 1987 to 2020?

## 0. Setup

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import (
    RAW_CSV, CLEANED_PARQUET, OUTPUT_DIR,
    DROP_COLS, CORE_COLS, INT_COLS,
    DELAY_MIN_MINUTES, DELAY_MAX_MINUTES, MIN_CARRIER_FLIGHTS,
    FIGURE_DPI, FIGURE_SIZE, COLOR_PRIMARY, COLOR_DELAY, COLOR_GOOD,
    TARGET_COL, MONTH_TO_SEASON, DOW_LABELS, MONTH_LABELS,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = FIGURE_SIZE
sns.set_theme(style='whitegrid')

print(f'Data file exists: {RAW_CSV.exists()}')
print(f'Output dir      : {OUTPUT_DIR}')

Data file exists: True
Output dir      : c:\Projects\bi\notebooks\..\output


## 1. Load Data

In [ ]:
df = pd.read_csv(RAW_CSV, low_memory=False)
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe4 in position 17667: invalid continuation byte

## 2. Initial Data Audit

In [ ]:
df.dtypes.to_frame('dtype').T

In [ ]:
missing = pd.DataFrame({
    'missing':     df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2),
}).sort_values('missing_pct', ascending=False)

missing[missing.missing > 0]

In [ ]:
top_missing = missing[missing.missing_pct > 0].head(20)
top_missing['missing_pct'].plot(kind='barh', color=COLOR_PRIMARY)
plt.xlabel('% Missing')
plt.title('Missing Values by Column')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'missing_values.png', dpi=FIGURE_DPI)
plt.show()

## 3. Preprocessing

In [ ]:
# DECISION 1: Drop redundant / all-null columns (defined in config.py)
df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)
print(f'After dropping redundant columns: {df.shape[1]} columns remain')

In [ ]:
# DECISION 2: Drop rows missing core identifiers (defined in config.py)
before = len(df)
df.dropna(subset=CORE_COLS, inplace=True)
print(f'Dropped {before - len(df):,} rows missing core identifiers')

In [ ]:
# DECISION 3: Remove physically impossible delay values (thresholds in config.py)
before = len(df)
df = df[
    (df['ArrDelay'].isna() | df['ArrDelay'].between(DELAY_MIN_MINUTES, DELAY_MAX_MINUTES)) &
    (df['DepDelay'].isna() | df['DepDelay'].between(DELAY_MIN_MINUTES, DELAY_MAX_MINUTES))
]
print(f'Removed {before - len(df):,} rows with impossible delay values')

In [ ]:
# DECISION 4: Cast column types (lists defined in config.py)
for col in INT_COLS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

df['FlightDate'] = pd.to_datetime(df['FlightDate'], errors='coerce')
print('Types corrected.')

In [ ]:
# DECISION 5: Feature engineering (mappings defined in config.py)
df['Decade']    = (df['Year'] // 10 * 10).astype('Int64')
df['Season']    = df['Month'].map(MONTH_TO_SEASON)
df['IsWeekend'] = df['DayOfWeek'].isin([6, 7]).astype(int)
df['DepHour']   = (df['CRSDepTime'] // 100).astype('Int64')
df[TARGET_COL]  = (df['ArrDel15'] == 1).astype(int)

print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 4. Exploratory Data Analysis

In [ ]:
df[['ArrDelay', 'DepDelay', 'Distance', 'AirTime', 'TaxiOut', 'TaxiIn']].describe().round(2)

In [ ]:
# Flight volume over time
df.groupby('Year').size().plot(kind='area', alpha=0.4, color=COLOR_PRIMARY)
plt.title('Number of Flights per Year (sample)')
plt.ylabel('Flight count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'flights_per_year.png', dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Delay rate over time
delay_rate = df.groupby('Year')[TARGET_COL].mean() * 100
delay_rate.plot(marker='o', ms=4, color=COLOR_DELAY)
plt.axhline(delay_rate.mean(), ls='--', color='gray', label='Overall avg')
plt.title('% Flights Delayed 15+ min per Year')
plt.ylabel('Delay rate (%)')
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delay_rate_by_year.png', dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Seasonality — delay by month
df.groupby('Month')['ArrDelay'].mean().plot(kind='bar', color=COLOR_PRIMARY)
plt.title('Average Arrival Delay by Month')
plt.xlabel('Month')
plt.ylabel('Avg delay (min)')
plt.xticks(range(12), MONTH_LABELS, rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delay_by_month.png', dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Delay by day of week
dow = (df.groupby('DayOfWeek')[TARGET_COL].mean() * 100).rename(index=DOW_LABELS)
dow.plot(kind='bar', color=COLOR_GOOD)
plt.title('Delay Rate by Day of Week')
plt.ylabel('Delay rate (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delay_by_dow.png', dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Airline delay rates (MIN_CARRIER_FLIGHTS threshold from config.py)
airline = (
    df.groupby('Reporting_Airline')
    .agg(flights=(TARGET_COL,'count'), delay_rate=(TARGET_COL,'mean'))
    .query(f'flights > {MIN_CARRIER_FLIGHTS}')
    .assign(delay_rate=lambda x: x.delay_rate * 100)
    .sort_values('delay_rate')
    .tail(10)
)
airline['delay_rate'].plot(kind='barh', color=COLOR_DELAY)
plt.title('Airline Delay Rate — Top 10 Worst')
plt.xlabel('Delay rate (%)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'delay_by_airline.png', dpi=FIGURE_DPI)
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['DepDelay','ArrDelay','Distance','AirTime','TaxiOut','TaxiIn',
                'Month','DayOfWeek','DepHour']
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_heatmap.png', dpi=FIGURE_DPI)
plt.show()

## 5. Save Cleaned Dataset

In [ ]:
df.to_parquet(CLEANED_PARQUET, index=False)
size_mb = CLEANED_PARQUET.stat().st_size / 1024 / 1024
print(f'Saved: {CLEANED_PARQUET.name} ({size_mb:.1f} MB)')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 6. Preprocessing Decisions Summary

| # | Decision | What | Why |
|---|----------|------|-----|
| 1 | Drop redundant columns | Airport seq IDs, WAC codes, Div3-5 | 100% null or duplicated by simpler columns |
| 2 | Drop rows missing core fields | Year, Month, Origin, Dest, Airline | Cannot assign a flight to any meaningful record without these |
| 3 | Remove impossible delays | Outside −6h / +24h | BTS data entry errors; <0.01% of rows |
| 4 | Delay cause nulls retained | CarrierDelay etc. ~89% null | Only populated for delayed flights since 2003 — structurally correct |
| 5 | Tail number nulls retained | ~20% missing | Pre-1995 reporting did not require tail numbers |
| 6 | Engineered: Decade, Season | Time buckets | Enable decade-level trends and seasonal analysis |
| 7 | Engineered: IsWeekend, DepHour | Demand features | Capture peak travel patterns |
| 8 | Engineered: IsDelayed | Binary target | Clean label for classification models (Assessment 3) |